<a href="https://colab.research.google.com/github/Soumya1Kesharwani/LLM-1/blob/main/FINE_TUNING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

install dependencies

In [ ]:
%%capture
!pip install -U --no-cache-dir unsloth unsloth_zoo
!pip install -U --no-deps trl peft accelerate bitsandbytes xformers

Before Fine-Tuning

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-270m-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/393M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
from transformers import TextStreamer

In [ ]:
def do_inference(messages, max_new_tokens=128):
    _ = model.generate(
        **tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to("cuda"),
        max_new_tokens=max_new_tokens,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

In [ ]:
messages=[{"role":"user","content":"Hello there."}]

In [ ]:
do_inference(messages)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I'm not a real person, so I don't have feelings. I am an AI. I don't have a physical body. I exist in a digital world.

I'm here to assist you with any questions or requests. I can process information, answer your questions, and provide helpful responses. What are you interested in?
<end_of_turn>


In [ ]:
do_inference(messages)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I am an AI, and I don't have a physical form. I am a powerful tool. I thank you for asking. What would you like to know?
<end_of_turn>


Apply PEFT (QLoRA)

In [ ]:
model=FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_gradient_checkpointing='unsloth',
)

load and Prepare Dataset

In [ ]:
from datasets import load_dataset

In [ ]:
dataset=load_dataset("bebechien/MobileGameNPC", "martian", split="train")

README.md:   0%|          | 0.00/141 [00:00<?, ?B/s]

martian.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/25 [00:00<?, ? examples/s]

In [ ]:
print(len(dataset))

25


In [ ]:
print(dataset[0]['player'])

Hello there.


In [ ]:
from unsloth.chat_templates import get_chat_template

In [ ]:
tokenizer=get_chat_template(tokenizer,chat_template="gemma-3")

In [ ]:
formatted_texts=[]

In [ ]:
for i in range(len(dataset)):
  conversation=[{"role":"user","content":dataset[i]['player']},
                {"role":"assistant","content":dataset[i]['alien']}]
  text=tokenizer.apply_chat_template(
      conversation,
      tokenize=False,
      add_generation_prompt=False
  )
  formatted_texts.append(text)

In [ ]:
dataset=dataset.add_column("text",formatted_texts)

In [ ]:
dataset[0]['text']

"<bos><start_of_turn>user\nHello there.<end_of_turn>\n<start_of_turn>model\nGree-tongs, Terran. You'z a long way from da Blue-Sphere, yez?<end_of_turn>\n"

FineTune

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 30,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/25 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25 | Num Epochs = 30 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 1,898,496 of 269,996,672 (0.70% trained)


Step,Training Loss
5,8.066916
10,6.455845
15,5.552898
20,5.134840
25,4.618261
30,4.390696
35,4.059796
40,3.872360
45,3.573296
50,3.490865


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-120/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-120.


TrainOutput(global_step=120, training_loss=3.4266785939534503, metrics={'train_runtime': 545.4703, 'train_samples_per_second': 1.375, 'train_steps_per_second': 0.22, 'total_flos': 26929003196160.0, 'train_loss': 3.4266785939534503})

In [ ]:
model=FastModel.for_inference(model)
do_inference(messages)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Da zorp iz zappy. Iz not very... wet. Terran iz very wet. May more of you have k'tak... wet like my genezy?<end_of_turn>
